## Amazon Sentiment Modeling 

This notebook builds a baseline sentiment classification model using Amazon product reviews. We train a logistic regression model with TF-IDF and metadata features, evaluate its performance, and compare it to a random forest model.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, isnan, isnull, mean, stddev, min, max, length, avg, expr, when, desc, concat, lit, coalesce
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pyspark.sql import Window
import numpy as np
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.classification import RandomForestClassifier

In [2]:
spark = SparkSession.builder \
    .appName("Amazon Reviews Training") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "6g") \
    .config("spark.executor.cores", "1") \
    .config("spark.default.parallelism", "16") \
    .config("spark.sql.shuffle.partitions", "16") \
    .getOrCreate()


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/03 16:41:15 INFO SparkEnv: Registering MapOutputTracker
25/04/03 16:41:15 INFO SparkEnv: Registering BlockManagerMaster
25/04/03 16:41:15 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
25/04/03 16:41:15 INFO SparkEnv: Registering OutputCommitCoordinator


In [3]:
amazon_dataset = spark.read.parquet("gs://final-project-bucket-amazon/processed/final_dataset.parquet")
print(f"Loaded {amazon_dataset.count()} reviews from parquet")

Loaded 18110850 reviews from parquet


In [4]:
# List of all feature columns
additional_features = [
    "helpful_ratio",
    "has_votes",
    "vine_binary",
    "verified_purchase_binary",
    "product_category_vec",
    "review_year",
    "review_month",
    "review_dayofweek"
]

# Combine 'features' (TF-IDF) and all other features
assembler = VectorAssembler(
    inputCols=["features"] + additional_features,
    outputCol="final_features"
)

# Transform dataset
assembled_df = assembler.transform(amazon_dataset)


In [5]:
# First: Split into train+val and test (85% / 15%)
train_val_df, test_df = assembled_df.randomSplit([0.85, 0.15], seed=42)

# Then: Split train+val into train and validation (70% / 15%)
train_df, val_df = train_val_df.randomSplit([0.8235, 0.1765], seed=42)

# Show sizes
print(f"Train count: {train_df.count()}")
print(f"Validation count: {val_df.count()}")
print(f"Test count (blind set): {test_df.count()}")


Train count: 12675558


Validation count: 2717659


Test count (blind set): 2717633


In [6]:
# Check for nulls in the training set
train_df.select([count(when(col(c).isNull(), c)).alias(c) for c in train_df.columns]).show()

+-------------+---------+-----------+------------------------+--------------------+-----------+------------+----------------+--------+-----+--------------+
|helpful_ratio|has_votes|vine_binary|verified_purchase_binary|product_category_vec|review_year|review_month|review_dayofweek|features|label|final_features|
+-------------+---------+-----------+------------------------+--------------------+-----------+------------+----------------+--------+-----+--------------+
|            0|        0|          0|                       0|                   0|          0|           0|               0|       0|    0|             0|
+-------------+---------+-----------+------------------------+--------------------+-----------+------------+----------------+--------+-----+--------------+



In [13]:
# Count number of instances per class
label_counts = train_df.groupBy("label").count().toPandas()

# Extract counts
neg_count = label_counts[label_counts['label'] == 0]['count'].values[0]
pos_count = label_counts[label_counts['label'] == 1]['count'].values[0]
total = neg_count + pos_count

# Compute weights: inverse of class frequency
neg_weight = total / (2 * neg_count)
pos_weight = total / (2 * pos_count)

print(f"Negative class weight: {neg_weight}")
print(f"Positive class weight: {pos_weight}")


Negative class weight: 2.150706384746216
Positive class weight: 0.6514503138233367


In [15]:
# Apply weights to train, val, and test sets
train_df = train_df.withColumn(
    "class_weight",
    when(col("label") == 0, lit(neg_weight)).otherwise(lit(pos_weight))
)

val_df = val_df.withColumn(
    "class_weight",
    when(col("label") == 0, lit(neg_weight)).otherwise(lit(pos_weight))
)

test_df = test_df.withColumn(
    "class_weight",
    when(col("label") == 0, lit(neg_weight)).otherwise(lit(pos_weight))
)


In [16]:
lr_weighted = LogisticRegression(
    featuresCol="final_features",
    labelCol="label",
    predictionCol="prediction",
    weightCol="class_weight",
    maxIter=20,
    regParam=0.1,
    elasticNetParam=1.0  # L1 regularization
)

lr_model = lr_weighted.fit(train_df)

In [17]:
# Function to evaluate a dataset
def evaluate_model(df, dataset_name):
    evaluator_acc = MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="accuracy")

    evaluator_f1 = MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="f1")

    acc = evaluator_acc.evaluate(df)
    f1 = evaluator_f1.evaluate(df)

    print(f"\nEvaluation on {dataset_name}")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")

# Run predictions using the weighted logistic regression model
train_predictions = lr_model.transform(train_df)
val_predictions = lr_model.transform(val_df)
test_predictions = lr_model.transform(test_df)

# Evaluate on all three sets
evaluate_model(train_predictions, "Training Set")
evaluate_model(val_predictions, "Validation Set")
evaluate_model(test_predictions, "Blind Test Set")



Evaluation on Training Set
Accuracy: 0.5206
F1 Score: 0.5420



Evaluation on Validation Set
Accuracy: 0.5209
F1 Score: 0.5422



Evaluation on Blind Test Set
Accuracy: 0.5209
F1 Score: 0.5424


In [19]:
# Create confusion matrix table
conf_matrix = test_predictions.groupBy("label", "prediction").count().orderBy("label", "prediction")
conf_matrix.show()


+-----+----------+-------+
|label|prediction|  count|
+-----+----------+-------+
|    0|       0.0| 568406|
|    0|       1.0|  63847|
|    1|       0.0|1238078|
|    1|       1.0| 847302|
+-----+----------+-------+



In [18]:
# Count number of rows for each label (0 and 1)
label_counts = train_df.groupBy("label").count().collect()

# Extract counts from Spark Row objects
for row in label_counts:
    if row["label"] == 0:
        neg_count = row["count"]
    else:
        pos_count = row["count"]

print(f"Negative: {neg_count}, Positive: {pos_count}")

Negative: 2946836, Positive: 9728722


In [20]:
# Calculate downsample ratio
downsample_ratio = neg_count / pos_count

# Filter each class
neg_class_df = train_df.filter(col("label") == 0)
pos_class_df = train_df.filter(col("label") == 1)

# Downsample the positive class
pos_class_downsampled = pos_class_df.sample(withReplacement=False, fraction=downsample_ratio, seed=42)

# Combine into balanced training set
train_balanced_df = neg_class_df.union(pos_class_downsampled)

print(f"Balanced training data count: {train_balanced_df.count()}")


Balanced training data count: 5896901


In [21]:
# Train with fewer trees & shallower depth for speed
rf = RandomForestClassifier(
    featuresCol="final_features",
    labelCol="label",
    predictionCol="prediction",
    numTrees=30,       
    maxDepth=8,   
    seed=42
) 

# Train on the downsampled training set
rf_model = rf.fit(train_balanced_df)


25/03/29 21:20:33 WARN DAGScheduler: Broadcasting large task binary with size 1124.6 KiB
25/03/29 21:33:13 WARN DAGScheduler: Broadcasting large task binary with size 1145.9 KiB
25/03/29 21:35:00 WARN DAGScheduler: Broadcasting large task binary with size 1185.8 KiB
25/03/29 21:36:48 WARN DAGScheduler: Broadcasting large task binary with size 1264.5 KiB
25/03/29 21:38:40 WARN DAGScheduler: Broadcasting large task binary with size 1410.4 KiB
25/03/29 21:40:33 WARN DAGScheduler: Broadcasting large task binary with size 1659.6 KiB
25/03/29 21:42:30 WARN DAGScheduler: Broadcasting large task binary with size 2.0 MiB
25/03/29 21:44:27 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB


In [22]:
# Run predictions on all 3 sets
rf_train_preds = rf_model.transform(train_df)
rf_val_preds = rf_model.transform(val_df)
rf_test_preds = rf_model.transform(test_df)

In [23]:
evaluate_model(rf_train_preds, "Random Forest - Training Set")
evaluate_model(rf_val_preds, "Random Forest - Validation Set")
evaluate_model(rf_test_preds, "Random Forest - Blind Test Set")

25/03/29 21:50:31 WARN DAGScheduler: Broadcasting large task binary with size 1393.9 KiB
25/03/29 21:53:09 WARN DAGScheduler: Broadcasting large task binary with size 1393.9 KiB
25/03/29 21:55:35 WARN DAGScheduler: Broadcasting large task binary with size 1393.9 KiB



Evaluation on Random Forest - Training Set
Accuracy: 0.7627
F1 Score: 0.7755


25/03/29 21:57:11 WARN DAGScheduler: Broadcasting large task binary with size 1393.9 KiB
25/03/29 21:58:48 WARN DAGScheduler: Broadcasting large task binary with size 1378.5 KiB



Evaluation on Random Forest - Validation Set
Accuracy: 0.7623
F1 Score: 0.7751


25/03/29 22:00:03 WARN DAGScheduler: Broadcasting large task binary with size 1378.5 KiB



Evaluation on Random Forest - Blind Test Set
Accuracy: 0.7625
F1 Score: 0.7753


In [24]:
rf_test_preds.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

25/03/29 22:04:00 WARN DAGScheduler: Broadcasting large task binary with size 1372.4 KiB
25/03/29 22:05:37 WARN DAGScheduler: Broadcasting large task binary with size 1334.0 KiB


+-----+----------+-------+
|label|prediction|  count|
+-----+----------+-------+
|    0|       0.0| 446618|
|    0|       1.0| 185635|
|    1|       0.0| 459685|
|    1|       1.0|1625695|
+-----+----------+-------+



In [4]:
spark.stop()

### Result Discussion

Logistic regression and random forest were both trained using techniques to address class imbalance. For logistic regression, class weights were applied to give more importance to the minority class (negative reviews). For random forest, the training data was balanced by downsampling the majority class (positive reviews) to match the number of negative reviews. Logistic regression achieved around 52% accuracy and 54% F1 score across all sets, indicating limited ability to capture both classes effectively. In contrast, the random forest model performed significantly better, achieving approximately 76.2% accuracy and 77.5% F1 score on training, validation, and test sets. It also produced a more balanced confusion matrix, predicting both positive and negative sentiment more reliably. Based on these results, random forest is the stronger model for this classification task.